In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Set random seed
np.random.seed(42)

In [ ]:
# Parameters
n_samples = 1470

In [ ]:
# Feature Generation
age = np.random.randint(22, 60, size=n_samples)
monthly_income = np.random.randint(2000, 20000, size=n_samples)
distance_from_home = np.random.randint(1, 30, size=n_samples)
total_working_years = np.abs(age - 20 + np.random.normal(0, 2, n_samples)).astype(int)
years_at_company = np.array([np.random.randint(0, min(15, twy + 1)) for twy in total_working_years])
overtime = np.random.choice(['Yes', 'No'], size=n_samples, p=[0.3, 0.7])
job_satisfaction = np.random.randint(1, 5, size=n_samples)
job_level = np.random.randint(1, 6, size=n_samples)
marital_status = np.random.choice(['Single', 'Married', 'Divorced'], size=n_samples)

In [ ]:
# Simple Attrition Logic (Influenced by Overtime and Income)
attrition_prob = (
    (overtime == 'Yes') * 0.4 +
    (monthly_income < 5000) * 0.3 +
    (job_satisfaction == 1) * 0.2 +
    (years_at_company < 2) * 0.1
)
attrition_prob = np.clip(attrition_prob, 0, 1)
attrition = np.random.binomial(1, attrition_prob)

In [ ]:
df = pd.DataFrame({
    'Age': age,
    'MonthlyIncome': monthly_income,
    'DistanceFromHome': distance_from_home,
    'TotalWorkingYears': total_working_years,
    'YearsAtCompany': years_at_company,
    'OverTime': overtime,
    'JobSatisfaction': job_satisfaction,
    'JobLevel': job_level,
    'MaritalStatus': marital_status,
    'Attrition': np.where(attrition == 1, 'Yes', 'No')
})

In [ ]:
# Save Raw Data
os.makedirs('07_employee_attrition/data', exist_ok=True)
df.to_csv('07_employee_attrition/data/hr_attrition_simulated.csv', index=False)

In [ ]:
# Preprocessing
X = df.drop('Attrition', axis=1)
y = (df['Attrition'] == 'Yes').astype(int)

In [ ]:
# Categorical Encoding
le_dict = {}
cat_cols = ['OverTime', 'MaritalStatus']
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    le_dict[col] = le

In [ ]:
os.makedirs('07_employee_attrition/models', exist_ok=True)
with open('07_employee_attrition/models/le_dict.pkl', 'wb') as f:
    pickle.dump(le_dict, f)

In [ ]:
with open('07_employee_attrition/models/feature_columns.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)

In [ ]:
print("Project 07 HR Data Successfully Simulated.")